# 07 — Activator Alerts & Operations Automation
**Real-Time Alerting for Fleet Operations**

Evaluates Gold-layer data against alert thresholds, writes an operational
`alert_configuration` and `alert_log` Delta table, then provides step-by-step
Fabric Activator (Reflex) wiring instructions.

| Alert | Activator Condition | Severity | Action |
|-------|---------------------|----------|--------|
| Empty Station | No_Bikes == 0 | Critical | Teams → #ops-dispatch |
| Full Station | No_Empty_Docks == 0 | Critical | Teams → #ops-dispatch |
| Low Availability | No_Bikes < 3 | Warning | Email → ops-planning |
| High Demand | No_Empty_Docks < 3 | Warning | Teams → #ops-planning |
| Demand Spike (ML) | predicted_demand ≥ 80th pctile | Warning | NB07 alert_log |
| Fleet Imbalance | neighbourhood util > 0.90 | Critical | NB07 alert_log |
| SLA Breach Risk | availability SLA < 95 % | Critical | NB07 alert_log |

> **Note:** Activator handles Rules 1-4 (simple column conditions).
> Rules 5-7 require computed metrics — evaluated by this notebook.

### Prerequisites
1. Attach **bicycles_gold** as the default lakehouse

2. Add **bicycles_silver** as an additional lakehouse4. (Optional) Run **NB06** first for forecast-based alerts
3. Run **NB03 → NB04** first so Gold tables exist

In [ ]:
# ============================================================
# CELL 1 — Configuration · Imports · Alert Thresholds
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType,
)
from pyspark.sql.functions import (
    col, when, current_timestamp, lit, count, avg as spark_avg,
    sum as spark_sum, round as spark_round, expr,
)
from datetime import datetime

# ── Cross-lakehouse configuration ─────────────────────────────
# Three-part names: {LAKEHOUSE}.dbo.{table}
SILVER_LAKEHOUSE = "bicycles_silver"
GOLD_LAKEHOUSE   = "bicycles_gold"

# ── Read Silver tables ────────────────────────────────────────────
def read_silver(table_name):
    return spark.sql(f"SELECT * FROM {SILVER_LAKEHOUSE}.{table_name}")

# ── Alert rule definitions ────────────────────────────────────
alert_schema = StructType([
    StructField("alert_id",       StringType(),  False),
    StructField("alert_name",     StringType(),  False),
    StructField("severity",       StringType(),  False),
    StructField("metric",         StringType(),  False),
    StructField("operator",       StringType(),  False),
    StructField("threshold",      DoubleType(),  False),
    StructField("window_minutes", IntegerType(), False),
    StructField("action",         StringType(),  False),
    StructField("kql_query_name", StringType(),  True),
    StructField("enabled",        BooleanType(), False),
])

# NOTE: thresholds are on the 0–1 utilization scale
alert_rules = [
    ("ALT-001", "Empty Station",       "critical", "No_Bikes",          "==", 0.0,   5,  "dispatch_truck",       "rebalance_alerts",      True),
    ("ALT-002", "Full Station",         "critical", "No_Empty_Docks",    "==", 0.0,   5,  "reroute_riders",       "rebalance_alerts",      True),
    ("ALT-003", "Low Availability",      "warning",  "No_Bikes",          "<",  3.0,   0,  "add_bikes",            "live_station_status",   True),
    ("ALT-004", "High Demand",          "warning",  "No_Empty_Docks",    "<",  3.0,   0,  "monitor_rebalance",    "live_station_status",   True),
    ("ALT-005", "Demand Spike (ML)",    "warning",  "predicted_demand",  ">=", 80.0,  60, "pre_position_bikes",   "hourly_demand_trend",   True),
    ("ALT-006", "Fleet Imbalance",      "critical", "neighbourhood_util",">=", 0.90,  15, "emergency_rebalance",  "neighbourhood_heatmap", True),
    ("ALT-007", "SLA Breach Risk",      "critical", "sla_pct",           "<",  95.0,  60, "escalate_ops_manager", "sla_monitoring",        True),
    ("ALT-008", "Anomaly Detected",     "warning",  "anomaly_score",     ">=", 2.0,   5,  "investigate",          "anomaly_detection",     True),
]

df_alerts = spark.createDataFrame(alert_rules, schema=alert_schema)

df_alerts.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.alert_configuration")
print(f"✓ alert_configuration: {df_alerts.count()} rules written")

In [ ]:
# ============================================================
# CELL 2 — Evaluate Station-Level Alerts (from Gold)
# ============================================================
# Uses fact_rebalancing (already filtered to problem stations) and
# falls back to silver_station_profile for a full sweep.

# ── Load Gold fact tables ─────────────────────────────────────
df_rebalancing = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.fact_rebalancing")
df_stations    = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.dim_station")
df_fact_avail  = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.fact_availability")

# ── ALT-001 / ALT-002: Empty & Full stations ─────────────────
empty_stations = df_rebalancing.filter(col("bikes_available") == 0)
empty_count    = empty_stations.count()

full_stations  = df_rebalancing.filter(col("empty_docks") == 0)
full_count     = full_stations.count()

# ── ALT-003 / ALT-004: Utilization extremes ──────────────────
# (utilization_pct is 0–1 scale everywhere)
high_util = df_stations.filter(col("current_utilization_pct") >= 0.85)
high_count = high_util.count()

low_util = df_stations.filter(col("current_utilization_pct") <= 0.15)
low_count = low_util.count()

# ── ALT-006: Neighbourhood fleet imbalance ───────────────────
df_neigh = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.dim_neighbourhood")
imbalanced_hoods = df_neigh.filter(col("neighbourhood_utilization_pct") >= 0.90)
imbalance_count  = imbalanced_hoods.count()

# ── ALT-007: SLA proxy — % of stations in "normal" status ────
total_stations = df_stations.count()
normal_stations = df_stations.filter(col("current_status") == "normal").count()
sla_pct = round(normal_stations / max(total_stations, 1) * 100, 1)
sla_breach = sla_pct < 95.0

print("=" * 60)
print("ALERT EVALUATION SUMMARY")
print("=" * 60)
print(f"  ALT-001 Empty Station:      {empty_count:>5} triggered  {'🔴 CRITICAL' if empty_count > 0 else '🟢 OK'}")
print(f"  ALT-002 Full Station:       {full_count:>5} triggered  {'🔴 CRITICAL' if full_count > 0 else '🟢 OK'}")
print(f"  ALT-003 High Utilization:   {high_count:>5} triggered  {'🟡 WARNING' if high_count > 0 else '🟢 OK'}")
print(f"  ALT-004 Low Utilization:    {low_count:>5} triggered  {'🟡 WARNING' if low_count > 0 else '🟢 OK'}")
print(f"  ALT-006 Fleet Imbalance:    {imbalance_count:>5} triggered  {'🔴 CRITICAL' if imbalance_count > 0 else '🟢 OK'}")
print(f"  ALT-007 SLA Compliance:     {sla_pct:>5.1f}%           {'🔴 BREACH' if sla_breach else '🟢 OK'}")
print(f"\n  Fleet SLA: {sla_pct}% ({normal_stations}/{total_stations} stations normal)")

In [ ]:
# ============================================================
# CELL 3 — Build Alert Log (Station-Level Events)
# ============================================================

alert_log_schema = StructType([
    StructField("alert_id",           StringType()),
    StructField("alert_name",         StringType()),
    StructField("severity",           StringType()),
    StructField("bikepoint_id",       StringType()),
    StructField("street",             StringType()),
    StructField("neighbourhood",      StringType()),
    StructField("recommended_action", StringType()),
    StructField("message",            StringType()),
    StructField("alert_time",         TimestampType()),
    StructField("acknowledged",       BooleanType()),
])

alert_rows = []
now = datetime.utcnow()

# ── Empty stations ────────────────────────────────────────────
if empty_count > 0:
    for row in empty_stations.select(
        "station_id", "neighbourhood"
    ).distinct().collect()[:50]:
        alert_rows.append((
            "ALT-001", "Empty Station", "critical",
            row["station_id"], None, row["neighbourhood"],
            "dispatch_truck",
            f"Station {row['station_id']} has 0 bikes — dispatch rebalancing truck",
            now, False,
        ))

# ── Full stations ─────────────────────────────────────────────
if full_count > 0:
    for row in full_stations.select(
        "station_id", "neighbourhood"
    ).distinct().collect()[:50]:
        alert_rows.append((
            "ALT-002", "Full Station", "critical",
            row["station_id"], None, row["neighbourhood"],
            "reroute_riders",
            f"Station {row['station_id']} has 0 empty docks — reroute riders",
            now, False,
        ))

# ── Neighbourhood imbalance ──────────────────────────────────
if imbalance_count > 0:
    for row in imbalanced_hoods.select(
        "neighbourhood_name", "neighbourhood_utilization_pct"
    ).collect():
        pct = round(row["neighbourhood_utilization_pct"] * 100, 1)
        alert_rows.append((
            "ALT-006", "Fleet Imbalance", "critical",
            None, None, row["neighbourhood_name"],
            "emergency_rebalance",
            f"{row['neighbourhood_name']} utilization at {pct}% — emergency rebalance",
            now, False,
        ))

# ── SLA breach ────────────────────────────────────────────────
if sla_breach:
    alert_rows.append((
        "ALT-007", "SLA Breach Risk", "critical",
        None, None, "FLEET-WIDE",
        "escalate_ops_manager",
        f"Fleet SLA at {sla_pct}% (target 95%) — escalate to ops manager",
        now, False,
    ))

# ── Write alert log (overwrite — each run is a fresh evaluation) ─
alert_rows = alert_rows[:100]   # cap for demo

if alert_rows:
    df_alert_log = spark.createDataFrame(alert_rows, schema=alert_log_schema)
    df_alert_log.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.alert_log")
    print(f"✓ alert_log: {len(alert_rows)} alerts written")
    display(df_alert_log.orderBy("severity", "neighbourhood"))
else:
    # Write empty table so downstream doesn't break
    df_alert_log = spark.createDataFrame([], schema=alert_log_schema)
    df_alert_log.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.alert_log")
    print("✓ alert_log: 0 alerts — fleet is healthy!")

In [ ]:
# ============================================================
# CELL 4 — Forecast-Based Alerts (from NB06 ML Model)
# ============================================================

try:
    df_forecast = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.forecast_demand")
    forecast_count = df_forecast.count()
    print(f"Loaded forecast_demand: {forecast_count} rows")
except Exception:
    df_forecast = None
    print("forecast_demand not found — run NB06 first. Skipping ML alerts.")

if df_forecast is not None and forecast_count > 0:
    # ALT-005: Demand spike — predicted demand above 80th percentile
    p80 = df_forecast.approxQuantile("predicted_demand", [0.80], 0.01)[0]
    demand_spikes = df_forecast.filter(col("predicted_demand") >= p80)
    spike_count = demand_spikes.count()

    # Pre-positioning recommendations
    pre_pos = df_forecast.filter(col("pre_position_recommended") == True)
    pre_pos_count = pre_pos.count()

    print(f"\n  ALT-005 Demand Spikes (≥ p80 = {p80:.0f}): {spike_count}")
    print(f"  Pre-positioning recommended hours:     {pre_pos_count}")

    if spike_count > 0:
        print("\n  Top predicted demand spikes:")
        display(
            demand_spikes.select(
                "forecast_hour", "neighbourhood", "predicted_demand",
                "demand_tier", "pre_position_recommended",
            ).orderBy(col("predicted_demand").desc()).limit(15)
        )

    # Append forecast alerts to alert_log
    forecast_rows = []
    if spike_count > 0:
        for row in demand_spikes.select(
            "neighbourhood", "forecast_hour", "predicted_demand"
        ).collect()[:30]:
            forecast_rows.append((
                "ALT-005", "Demand Spike (ML)", "warning",
                None, None, row["neighbourhood"],
                "pre_position_bikes",
                f"Predicted demand {int(row['predicted_demand'])} at {row['forecast_hour']}",
                datetime.utcnow(), False,
            ))

    if forecast_rows:
        df_f_alerts = spark.createDataFrame(forecast_rows, schema=alert_log_schema)
        df_f_alerts.write.format("delta").mode("append") \
            .option("mergeSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.alert_log")
        print(f"\n✓ {len(forecast_rows)} forecast-based alerts appended to alert_log")

    # ── Summary ───────────────────────────────────────────────
    total_alerts = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.alert_log").collect()[0]["c"]
    print(f"\nTotal alerts in alert_log: {total_alerts}")
else:
    total_alerts = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.alert_log").collect()[0]["c"]
    print(f"\nTotal alerts in alert_log (station-level only): {total_alerts}")

In [ ]:
# ============================================================
# CELL 5 — Fabric Activator (Reflex) Setup Guide + Summary
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║           FABRIC ACTIVATOR (REFLEX) SETUP GUIDE             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Step 1: Create Activator in Fabric UI                       ║
║    Workspace → + New → Reflex (Activator)                    ║
║    Name: "BicycleFleet_Activator"                            ║
║                                                              ║
║  Step 2: Add Data Source                                     ║
║    Connect to Eventhouse: bikerentaleventhouse                ║
║    Database: bikerentaldb                                    ║
║    (This gives Activator real-time streaming data)           ║
║                                                              ║
║  Step 3: Create Objects (one per station metric)             ║
║    Object: "Station"                                         ║
║    Key column: BikepointID                                   ║
║    Properties: No_Bikes, No_Empty_Docks, Street,             ║
║                Neighbourhood                                 ║
║                                                              ║
║  Step 4: Define Trigger Rules                                ║
║  (Activator supports simple column conditions only —          ║
║   no computed columns, no duration windows)                   ║
║                                                              ║
║    ┌─ Rule 1: EMPTY STATION ─────────────────────────────┐   ║
║    │  Column:    No_Bikes                                 │   ║
║    │  Condition: Is equal to → 0                          │   ║
║    │  Action:    Send a Teams message                     │   ║
║    │  To:        #ops-dispatch                            │   ║
║    │  Message:   "🚨 Station {BikepointID} on {Street}   │   ║
║    │              is EMPTY — dispatch rebalancing truck"  │   ║
║    └─────────────────────────────────────────────────────┘   ║
║                                                              ║
║    ┌─ Rule 2: FULL STATION ──────────────────────────────┐   ║
║    │  Column:    No_Empty_Docks                           │   ║
║    │  Condition: Is equal to → 0                          │   ║
║    │  Action:    Send a Teams message                     │   ║
║    │  To:        #ops-dispatch                            │   ║
║    │  Message:   "🚨 Station {BikepointID} FULL —        │   ║
║    │              reroute riders to nearby stations"      │   ║
║    └─────────────────────────────────────────────────────┘   ║
║                                                              ║
║    ┌─ Rule 3: LOW AVAILABILITY ──────────────────────────┐   ║
║    │  Column:    No_Bikes                                 │   ║
║    │  Condition: Is less than → 3                         │   ║
║    │  Action:    Send an email                            │   ║
║    │  To:        ops-planning@company.com                 │   ║
║    │  Subject:   ⚠️ Low Bike Availability —               │   ║
║    │             Station {BikepointID}                    │   ║
║    │  Headline:  Station Running Low on Bikes             │   ║
║    │  Message:   Station {BikepointID} on {Street} in    │   ║
║    │             {Neighbourhood} has only {No_Bikes}      │   ║
║    │             bike(s) remaining. Dispatch additional   │   ║
║    │             bikes to prevent going empty.            │   ║
║    └─────────────────────────────────────────────────────┘   ║
║                                                              ║
║    ┌─ Rule 4: HIGH DEMAND ───────────────────────────────┐   ║
║    │  Column:    No_Empty_Docks                           │   ║
║    │  Condition: Is less than → 3                         │   ║
║    │  Action:    Send a Teams message                     │   ║
║    │  To:        #ops-planning                            │   ║
║    │  Message:   "⚠️ Station {BikepointID} on {Street}   │   ║
║    │              in {Neighbourhood} has only             │   ║
║    │              {No_Empty_Docks} empty dock(s) —        │   ║
║    │              redistribute bikes to nearby stations" │   ║
║    └─────────────────────────────────────────────────────┘   ║
║                                                              ║
║  Step 5: Test & Activate                                     ║
║    - Ensure Eventstream RTIbikeRental is ACTIVE              ║
║    - Verify data flows to bikerentaleventhouse               ║
║    - Start the Activator — monitor alerts for 5 minutes      ║
║    - Enable production rules once verified                   ║
║                                                              ║
║  Optional: Add forecast-based rules                          ║
║    - Connect second source: bicycles_gold lakehouse          ║
║    - Create rule on forecast_demand table                    ║
║    - Condition: predicted_demand > {p80 threshold}           ║
║    - Action: Pre-position bikes notification                 ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# ── Pipeline summary ──────────────────────────────────────────
print("=" * 65)
print("END-TO-END PIPELINE SUMMARY")
print("=" * 65)
print("""
  Eventstream RTIbikeRental (BicyclesRental sample)
       │
       ├──► bikerental_bronze_raw.bikeraw_tb  (auto-landed)
       │
       ├──► bikerentaleventhouse.bikerentaldb (real-time KQL)
       │         │
       │         └──► Activator Rules (Empty/Full/Low/High)
       │
       └──► NB03 Silver ──► NB04 Gold ──► Direct Lake ──► Power BI
                                │
                          NB06 ML Forecast

                                │print("\n✓ Activator notebook complete")

                          NB07 Alert Log ──► Activator (optional)

""")        print(f"  {tbl:<28} {'MISSING':>8}  {desc}")

    except Exception:

tables = {        print(f"  {tbl:<28} {n:>8,}  {desc}")

    "alert_configuration": "Activator rule definitions",        n = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.{tbl}").collect()[0]["c"]

    "alert_log":           "Triggered alert history",    try:

}for tbl, desc in tables.items():

print(f"\n{'Table':<30} {'Rows':>8}  Description")print("-" * 60)